## SETUP ##


In [13]:
import psycopg2
import pandas as pd

# Shared connection parameters — change these to match your setup
DB_CONFIG = {
    
    "host":     "localhost",
    "dbname": "bank_app_review",
    "user":     "ilham",
    "password": "12345678",
    "port": 5432
}

In [14]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- bank_app_review table ---
# Create bank_app_review table
# SERIAL auto-increments the ID on every insert — no need to supply it manually
# UNIQUE ensures no two restaurants share the same name
cur.execute("""
    CREATE TABLE IF NOT EXISTS bank_app_review (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE
    );
""")

# --- reviews table (must come after restaurants because of the foreign key) ---
# Create reviews table
# restaurant_id is a FOREIGN KEY — it must match an existing restaurants.restaurant_id
# This link is what allows us to JOIN the two tables later

cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id INT REFERENCES bank_app_review(bank_id),
        review_text   TEXT,
        rating        INT,
        review_date   DATE,
        source        VARCHAR(50)
    );
""")

conn.commit()
cur.close()
conn.close()

print("Tables created successfully.")

InsufficientPrivilege: permission denied for schema public
LINE 2:     CREATE TABLE IF NOT EXISTS bank_app_review (
                                       ^


In [15]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# 1. Create a custom schema and switch to it automatically
cur.execute("CREATE SCHEMA IF NOT EXISTS bank_data;")
cur.execute("SET search_path TO bank_data;")

# --- bank_app_review table ---
cur.execute("""
    CREATE TABLE IF NOT EXISTS bank_app_review (
        bank_id   SERIAL PRIMARY KEY,
        bank_name VARCHAR(255) UNIQUE
    );
""")

# --- reviews table ---
cur.execute("""
    CREATE TABLE IF NOT EXISTS reviews (
        review_id     SERIAL PRIMARY KEY,
        bank_id       INT REFERENCES bank_app_review(bank_id),
        review_text   TEXT,
        rating        INT,
        review_date   DATE,
        source        VARCHAR(50)
    );
""")

conn.commit()
cur.close()
conn.close()

print("Tables successfully created inside the custom 'bank_data' schema!")


Tables successfully created inside the custom 'bank_data' schema!


## LOAD CSV FILE TO PANDAS ##

In [20]:
df_banks= pd.read_csv("../data/banks.csv")
df_bank_review  = pd.read_csv("../data/bank_reviews.csv")

print(f"bank: {len(df_banks)} rows")
print(f"Reviews:     {len(df_bank_review)} rows")
print()
print(df_banks)
print()
print(df_bank_review)

FileNotFoundError: [Errno 2] No such file or directory: '../data/banks.csv'

In [21]:
import pandas as pd
import glob

# 1. Load the main bank lookup table
# This maps bank_id to bank_name
banks_df = pd.read_csv("banks.csv")
print("Successfully loaded banks.csv. Shape:", banks_df.shape)

Successfully loaded banks.csv. Shape: (3, 2)


In [18]:
import os
print("Python is looking inside:", os.getcwd())
print("Files Python can see:", os.listdir())


Python is looking inside: c:\Users\yared\Desktop\week2\notebooks
Files Python can see: ['abbysinia.ipynb', 'abbysinia_bank_reviews.csv', 'AWASH.ipynb', 'awash_bank_reviews.csv', 'bank_review.ipynb', 'CBE.ipynb', 'commercial_bank_reviews.csv', 'data', 'init.py', 'README.md', 'test_pipeline.py']


In [19]:
# Add a column indicating which bank the data belongs to before merging
# (Assuming your 3 individual files have columns like 'text', 'rating', 'date', 'source')
bank1_df['source'] = 'Bank 1 App'
bank2_df['source'] = 'Bank 2 App'
bank3_df['source'] = 'Bank 3 App'

# Combine all 3 sets into a single master review table
reviews_df = pd.concat([bank1_df, bank2_df, bank3_df], ignore_index=True)
print("Total rows combined from all 3 scrapers:", reviews_df.shape[0])


NameError: name 'bank1_df' is not defined

In [26]:
import pandas as pd
import glob

# Use the absolute Windows path to load the file explicitly
banks_df = pd.read_csv("banks.csv")
print("Successfully loaded banks.csv. Shape:", banks_df.shape)

# Load your 3 review files using explicit absolute path formatting
# (Change the names below to match your exact file names on your computer)
bank1_df = pd.read_csv("awash_bank_reviews.csv")
bank2_df = pd.read_csv("commercial_bank_reviews.csv")
bank3_df = pd.read_csv("abbysinia_bank_reviews.csv")

print("All individual DataFrames loaded successfully!")


Successfully loaded banks.csv. Shape: (3, 2)
All individual DataFrames loaded successfully!


In [31]:
# Add a column indicating which bank the data belongs to before merging
# (Assuming your 3 individual files have columns like 'text', 'rating', 'date', 'source')
bank1_df['source'] = 'Bank 1 App'
bank2_df['source'] = 'Bank 2 App'
bank3_df['source'] = 'Bank 3 App'

# Combine all 3 sets into a single master review table
reviews_df = pd.concat([bank1_df, bank2_df, bank3_df], ignore_index=True)
print("Total rows combined from all 3 scrapers:", reviews_df.shape[0])


Total rows combined from all 3 scrapers: 1500


In [28]:
print(reviews_df.head())

                               reviewId            userName  \
0  d251dd23-f516-4c34-8be9-2f6205ff04a1         Nyalite Tut   
1  911ed0ae-df98-4199-ac01-e8b91731cfe2        Solomon Wale   
2  3c6b9df9-ff20-4ccb-b918-4336a0cce1d6           Wende Man   
3  15364398-e025-435a-8fcc-b6ce863df75d          Abera Mamo   
4  f2afbfce-ee6d-4a33-8d20-a53a820c15d5  Dhaloota Boruu1122   

                                           userImage  \
0  https://play-lh.googleusercontent.com/a/ACg8oc...   
1  https://play-lh.googleusercontent.com/a-/ALV-U...   
2  https://play-lh.googleusercontent.com/a/ACg8oc...   
3  https://play-lh.googleusercontent.com/a-/ALV-U...   
4  https://play-lh.googleusercontent.com/a/ACg8oc...   

                                             content  score  thumbsUpCount  \
0                                   good application      5              0   
1                                      How i can use      5              0   
2                                Wende. Weju. Meku

In [32]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# --- Insert restaurants ---
for _, row in df_banks.iterrows():
    cur.execute(
        """
        INSERT INTO banks (bank_id, bank_name)
        VALUES (%s, %s)
        ON CONFLICT DO NOTHING;
        """,
        (int(row["bank_id"]), row["bank_name"])
    )

print(f"Inserted {len(df_reviews)} bank rows.")

# --- Insert reviews ---
for _, row in df_reviews.iterrows():
    cur.execute(
        """
        INSERT INTO reviews (review_id, bank_id, review_text, rating, review_date, source)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT DO NOTHING;
        """,
        (
            int(row["review_id"]),
            int(row["bank_id"]),
            row["review_text"],
            int(row["rating"]),
            row["review_date"],
            row["source"]
        )
    )

print(f"Inserted {len(df_reviews)} review rows.")

# --- Save and close ---
conn.commit()   # permanently write all inserts to disk
cur.close()
conn.close()

print("Done — connection closed.")

NameError: name 'df_banks' is not defined

In [35]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

# 1. CRUCIAL: Direct PostgreSQL to look inside your custom schema space
cur.execute("SET search_path TO bank_data;")

# --- Insert banks into bank_app_review ---
# Changed table name from 'banks' to 'bank_app_review' to match your schema blueprint
for _, row in df_reviews.iterrows():
    cur.execute(
        """
        INSERT INTO bank_app_review (bank_id, bank_name)
        VALUES (%s, %s)
        ON CONFLICT (bank_id) DO NOTHING;
        """,
        (int(row["bank_id"]), row["bank_name"])
    )

print(f"Processed {len(df_banks)} bank rows.")

# --- Insert reviews ---
# Note: Since review_id uses 'SERIAL', it will fail if you pass manual IDs that clash with the auto-sequencer.
# 'ON CONFLICT (review_id) DO NOTHING' handles any duplicate entries safely.
for _, row in df_reviews.iterrows():
    cur.execute(
        """
        INSERT INTO reviews (review_id, bank_id, review_text, rating, review_date, source)
        VALUES (%s, %s, %s, %s, %s, %s)
        ON CONFLICT (review_id) DO NOTHING;
        """,
        (
            int(row["review_id"]),
            int(row["bank_id"]),
            row["review_text"],
            int(row["rating"]),
            row["review_date"],
            row["source"]
        )
    )

print(f"Processed {len(df_reviews)} review rows.")

# --- Save and close ---
conn.commit()   # permanently write all inserts to disk
cur.close()
conn.close()

print("Done — connection closed.")


NameError: name 'df_reviews' is not defined

In [36]:
import psycopg2
from psycopg2.extras import execute_values

# 1. Establish your database connection
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()

try:
    # Activate your custom schema
    cur.execute("SET search_path TO bank_data;")
    
    # 2. STEP ONE: Insert the Bank Names into the parent table
    # Using 'ON CONFLICT DO NOTHING' ensures if you rerun this cell, it won't crash on duplicate bank names
    banks_to_insert = [("Commercial Bank of Ethiopia",), ("Awash Bank",), ("Bank of Abyssinia",)]
    
    execute_values(cur, """
        INSERT INTO bank_app_review (bank_name) 
        VALUES %s 
        ON CONFLICT (bank_name) DO NOTHING;
    """, banks_to_insert)
    
    # Fetch the auto-generated bank_ids from the database to create a Python lookup mapping
    cur.execute("SELECT bank_id, bank_name FROM bank_app_review;")
    bank_lookup = {name: b_id for b_id, name in cur.fetchall()}
    print("Database Bank Mapping:", bank_lookup)
    
    # 3. STEP TWO: Map the reviews DataFrame to match your database schema
    # Map your combined DataFrame column names to the database fields:
    # Example: If your dataframe column is 'review', map it to 'review_text'
    column_mapping = {
        'reviewDescription': 'review_text',  # <-- Change 'reviewDescription' to your actual column name
        'score': 'rating',                  # <-- Change 'score' to your actual column name
        'at': 'review_date',                # <-- Change 'at' to your actual column name
        'source': 'source'
    }
    
    # Map your DataFrames to include the correct numeric bank_id column using your source text
    # (Adjust 'Commercial Bank of Ethiopia', etc., to match whatever string is in your 'source' column)
    source_to_name = {
        'CBE': 'Commercial Bank of Ethiopia',
        'Awash': 'Awash Bank',
        'Abyssinia': 'Bank of Abyssinia'
    }
    
    # Process your reviews DataFrame to line up with the database structure
    # 1. Map source keys to canonical bank names, then map those names to database IDs
    # (Replace 'source' below if your column identifying the bank has a different header)
    reviews_df['bank_id'] = reviews_df['source'].map(source_to_name).map(bank_lookup)
    
    # Rename columns to match database names exactly
    db_ready_df = reviews_df.rename(columns=column_mapping)
    
    # Extract only the specific columns required by your SQL reviews table structure
    final_columns = ['bank_id', 'review_text', 'rating', 'review_date', 'source']
    insert_data = db_ready_df[final_columns].dropna(subset=['bank_id']).values.tolist()
    
    # 4. STEP THREE: Batch insert all 1,500 reviews efficiently
    print(f"Beginning batch insertion of {len(insert_data)} review records...")
    
    execute_values(cur, """
        INSERT INTO reviews (bank_id, review_text, rating, review_date, source)
        VALUES %s;
    """, insert_data)
    
    # Commit the transactions permanently to the disk
    conn.commit()
    print("🎉 All 1,500 records inserted successfully into PostgreSQL!")

except Exception as e:
    conn.rollback()
    print("❌ Database insertion failed. Transaction rolled back safely.")
    print("Error Details:", e)

finally:
    cur.close()
    conn.close()


Database Bank Mapping: {'Commercial Bank of Ethiopia': 1, 'Awash Bank': 2, 'Bank of Abyssinia': 3}
❌ Database insertion failed. Transaction rolled back safely.
Error Details: "['review_text'] not in index"
